# 02 – Results
Builds the charts used in the README from the local silver and gold layers.

Run first: `run_bronze`, `run_silver`, `run_gold`. Charts are saved to `docs/images/`.
Only pipeline-level counts are charted; suburb prices stay local (CC BY-NC-ND 4.0).

In [1]:
import sys
sys.path.append("../src")

import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql import functions as F

from pipeline.gold import to_dealings
from pipeline.spark import get_spark

spark = get_spark()
sales = spark.read.parquet("../data/silver/sales")
dealings = to_dealings(sales)

## Rows through the pipeline
Every bronze sale record must end up in exactly one place: silver, quarantine, or removed as a
duplicate. Gold then collapses multi-property sales into one row per dealing.

In [ ]:
bronze_rows = spark.read.parquet("../data/bronze/record_type=B").count()
quarantined = spark.read.parquet("../data/silver/quarantine").count()
silver_rows = sales.count()
steps = {
    "Bronze sale records": bronze_rows,
    "Quarantined": quarantined,
    "Duplicates removed": bronze_rows - quarantined - silver_rows,
    "Silver sales": silver_rows,
    "Gold dealings": dealings.count(),
}
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.barh(list(steps), list(steps.values()), color="#4C72B0")
for i, value in enumerate(steps.values()):
    ax.text(value, i, f" {value:,}", va="center")
ax.invert_yaxis()
ax.set_xlim(0, bronze_rows * 1.15)
ax.set_title("Rows through the pipeline (4 weekly files, Sep–Oct 2026)")
plt.tight_layout()
fig.savefig("../docs/images/pipeline_funnel.png", dpi=150)

## Late-arriving sales
Contract year of the sales published in four weekly files. The y-axis is log scale because
2026 has over 10,000 dealings and some earlier years have one.
This is why gold recomputes past quarters, not only the current one.

In [ ]:
by_year = pd.DataFrame([
    r.asDict() for r in dealings
    .groupBy(F.year("contract_date").alias("contract_year")).count()
    .orderBy("contract_year").collect()
])
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(by_year["contract_year"].astype(str), by_year["count"], color="#4C72B0")
ax.set_yscale("log")
ax.set_title("Contract year of sales published in 4 weekly files")
ax.set_xlabel("Contract year")
ax.set_ylabel("Dealings (log scale)")
plt.xticks(rotation=90)
plt.tight_layout()
fig.savefig("../docs/images/late_arriving_sales.png", dpi=150)